In [13]:
from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [14]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y      rotation
count  12860.000000   12860.000000  12860.000000
mean   35060.961230  128079.732587     25.250389
std      593.173520     951.762144     62.562695
min    33667.553332  125996.367175      0.000000
25%    34622.695128  127293.973600      0.000000
50%    35000.922088  128002.283775      0.000000
75%    35483.685992  128959.993661      0.000000
max    36625.891051  129787.710488    270.000000
<class 'pandas.DataFrame'>
RangeIndex: 12889 entries, 0 to 12888
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      12889 non-null  str    
 1   layer     12889 non-null  str    
 2   handle    12889 non-null  str    
 3   name      12860 non-null  str    
 4   X         12860 non-null  float64
 5   Y         12860 non-null  float64
 6   rotation  12860 non-null  float64
 7   UFV       12860 non-null  str    
 8   PST       12860 non-null  str    
 9   INV       12831 non-null  s

In [15]:
mask = (ELM_df['name'] == '1x96p_ponta') | (ELM_df['name'] == '1x96p_meio') | (ELM_df['name'] == '1x64p') | (ELM_df['name'] == '1x32p')
tracker_df = ELM_df[mask].copy()

tracker_df.drop(columns=['handle', 'type', 'QBT', 'DSJ', 'PVM', 'GRP'], inplace=True)

print(tracker_df)
print(tracker_df.describe())
print(tracker_df.info())

               layer   name             X              Y  rotation UFV PST INV
58     EMF_em_pst_21  1x64p  34979.686150  127465.060575     180.0   0  21   9
59     EMF_em_pst_21  1x64p  34979.686150  127507.562575       0.0   0  21   9
60     EMF_em_pst_21  1x64p  34985.986150  127465.060575     180.0   0  21   9
61     EMF_em_pst_21  1x64p  34985.986150  127507.562575       0.0   0  21   9
62     EMF_em_pst_21  1x64p  34992.286150  127465.060575     180.0   0  21   9
...              ...    ...           ...            ...       ...  ..  ..  ..
12884  EMF_em_pst_01  1x32p  35683.005250  129540.154750     180.0   0  01  12
12885  EMF_em_pst_01  1x32p  35689.305250  129540.154750     180.0   0  01  12
12886  EMF_em_pst_01  1x32p  35701.905250  129540.154750     180.0   0  01  12
12887  EMF_em_pst_01  1x32p  35708.205250  129540.154750     180.0   0  01  12
12888  EMF_em_pst_09  1x32p  34127.453648  128845.259495       0.0   0  09  04

[12367 rows x 8 columns]
                  X       

In [16]:
reinsert_mask = (ELM_df['name'] != '1x96p_ponta') & (ELM_df['name'] != '1x32p') & (ELM_df['rotation'] != 180)
reinsert_df = tracker_df[reinsert_mask].copy()

y_fix_64p = 21.251
y_fix_96p_to_64p = 21.264
y_fix_96p_to_32p = 43.503 + 21.264

reinsert_df['Y'] = np.where(
    reinsert_df['name'] == '1x64p',
    reinsert_df['Y']-y_fix_64p,
    reinsert_df['Y']-y_fix_96p_to_64p
)

reinsert_df['Y2'] = np.where(
    reinsert_df['name'] == '1x96p_meio',
    reinsert_df['Y']+y_fix_96p_to_32p,
    0
)

reinsert_df['new_name'] = 'EMF_tracker_64p'

reinsert_df['Script'] = np.where(
    reinsert_df['name'] == '1x96p_meio',
    reinsert_df.apply(
        lambda row:
        f"._insert {row['new_name']} {row['X']},{row['Y']} 1 1 0\n"
        f"._insert EMF_tracker_32p {row['X']},{row['Y2']} 1 1 0",
        axis=1
    ),
    reinsert_df.apply(
        lambda row:
        f"._insert {row['new_name']} {row['X']},{row['Y']} 1 1 0",
        axis=1
    )
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Reinserting Trackers\Reinserting Trackers 0B.scr", 'w') as f:
    for i in reinsert_df['layer'].unique():
        mask = reinsert_df['layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(reinsert_df[mask]['Script'].astype(str)) + '\n')



C:\Users\Usuario\AppData\Local\Temp\ipykernel_15996\2252478718.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reinsert_df = tracker_df[reinsert_mask].copy()
